# Khojki OCR: frozen evaluation
Prepared offline. No training. Upload paired ZIP using Files panel before running setup. Original model selected on development CER; held-out references fixed before predictions. Download the output archive and terminate the runtime. Minimum account reserve: 75 units.


In [ ]:
# Upload paired ZIP with Colab Files panel, then run this cell.
import os,sys,json,hashlib,zipfile,subprocess,time,platform
from pathlib import Path
START=time.monotonic()
BUNDLE=Path('/content/khojki-frozen-evaluation.zip')
assert BUNDLE.exists(), 'Upload the paired ZIP using the Files panel first.'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest()=='d678cb6419de29b3eac87b0df10346a9958ae3ce7684b4acf976e2ec411fc138'
ROOT=Path('/content/khojki-ocr')
with zipfile.ZipFile(BUNDLE) as z:z.extractall('/content')
for name,digest in json.loads((ROOT/'cloud-manifest.json').read_text()).items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest()==digest,name
os.environ['PYTHONPATH']=str(ROOT/'src')
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(ROOT)+'[test]'],check=True,timeout=180)
import torch
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0),torch.__version__)
OUT=ROOT/'results';OUT.mkdir(exist_ok=True)
def run(args,name,timeout=180):
    log=OUT/(name+'.log')
    with log.open('w') as f:p=subprocess.run([sys.executable,*map(str,args)],stdout=f,stderr=subprocess.STDOUT,timeout=timeout,cwd=ROOT)
    print(name,log.read_text()[-4000:]);p.check_returncode()
saved=torch.load(ROOT/'experiments/curve-10/best.pt',map_location='cpu',weights_only=False)
(ROOT/'models').mkdir(exist_ok=True)
MODEL=ROOT/'models/printed-khojki.pt'
torch.save({'model':saved['model'],'vocab':saved['vocab'],'height':48,'source_sha256':json.loads((ROOT/'experiments/final-freeze.json').read_text())['checkpoint_sha256']},MODEL)
del saved
(OUT/'environment.json').write_text(json.dumps({'python':sys.version,'torch':torch.__version__,'gpu':torch.cuda.get_device_name(0),'platform':platform.platform()},indent=2))
run(['-m','pytest','-q'],'tests')


In [ ]:
# Frozen model; no training or checkpoint selection in this notebook.
if not (OUT/'evaluation-complete.json').exists():
    run([ROOT/'scripts/evaluate.py','--checkpoint',MODEL,'--split','dev','--device','cuda','--output',OUT/'dev-crops.json'],'dev-crops')
    assert json.loads((OUT/'dev-crops.json').read_text())['summary']['errors']==10
    run([ROOT/'scripts/stress_eval.py','--checkpoint',MODEL,'--device','cuda','--output',OUT/'dev-stress.json'],'dev-stress')
    run([ROOT/'scripts/evaluate_pages.py','--checkpoint',MODEL,'--split','dev','--device','cuda','--output',OUT/'dev-pages'],'dev-pages')
    # Held-out test is evaluated once with the preselected model and settings.
    run([ROOT/'scripts/evaluate_pages.py','--checkpoint',MODEL,'--split','test','--device','cuda','--output',OUT/'test-pages'],'test-pages')
    # CPU deployment benchmark, same frozen predictions; not a model-selection loop.
    run(['-m','khojki_ocr',ROOT/'data/pages/p19.png','--checkpoint',MODEL,'--device','cpu','--output',OUT/'cpu-demo'],'cli-cpu')
    run(['-m','khojki_ocr',ROOT/'data/pages/p23.png','--checkpoint',MODEL,'--device','cuda','--output',OUT/'unscored-print'],'unscored-print')
    run(['-m','khojki_ocr',ROOT/'data/pages/p04.png','--checkpoint',MODEL,'--device','cuda','--output',OUT/'unscored-handwriting'],'unscored-handwriting')
    (OUT/'evaluation-complete.json').write_text(json.dumps({'session_seconds':time.monotonic()-START,'training_performed':False}))
else:print('Saved evaluation exists; skipping repeated evaluation.')
print('TEST',json.loads((OUT/'test-pages/evaluation.json').read_text())['summary'])


In [ ]:
# Export via visible notebook link; local copy must be hash-verified before disconnecting.
import base64
from IPython.display import HTML,display
RESULT=Path('/content/khojki-evaluation-results.zip')
subprocess.run([sys.executable,'-m','pip','freeze'],stdout=(OUT/'requirements-colab.txt').open('w'),check=True)
with zipfile.ZipFile(RESULT,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in ['results','models']:
        for p in (ROOT/folder).rglob('*'):
            if p.is_file():z.write(p,str(p.relative_to(ROOT)))
encoded=base64.b64encode(RESULT.read_bytes()).decode('ascii')
display(HTML('<a id="evaluation-export" download="khojki-evaluation-results.zip" href="data:application/zip;base64,'+encoded+'">Download frozen evaluation results and model</a>'))
print('RESULT_SHA256',hashlib.sha256(RESULT.read_bytes()).hexdigest(),'BYTES',RESULT.stat().st_size)
del encoded
# Verify downloaded file, then Runtime > Disconnect and delete runtime.
